# Silhouette Comparison — Method 1 vs Method 3b
**Why silhouette was not used for k-selection**

This notebook produces a single comparison figure showing silhouette behaviour
for both methods across all 12 cases (4 building types × 3 variables).

Key finding: silhouette returned k=2 in all 12 cases for Method 1.
For Method 3b, silhouette artificially inflates toward 1.0 for electricity.
Both behaviours confirm elbow is the correct metric for this data.

---
## Step 1 — Imports and Configuration

In [2]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

BASE_DIR   = os.getcwd()
FILE_M1    = os.path.join(BASE_DIR, 'DEA_method1_final.parquet')
FILE_M3B   = os.path.join(BASE_DIR, 'DEA_method3b_final.parquet')
OUTPUT_DIR = os.path.join(BASE_DIR, 'clustering_results')

K_MIN        = 2
K_MAX        = 10
SAMPLE_N     = 30_000
RANDOM_STATE = 42
SIZE_CLASSES = ['SFH', 'TH', 'MFH', 'AB']

print(f'BASE_DIR       : {BASE_DIR}')
print(f'FILE_M1 exists : {os.path.exists(FILE_M1)}')
print(f'FILE_M3B exists: {os.path.exists(FILE_M3B)}')
print('Configuration ready')

BASE_DIR       : c:\Users\zaito\Downloads\clustering
FILE_M1 exists : True
FILE_M3B exists: True
Configuration ready


---
## Step 2 — Load Both Parquets

In [3]:
BASE_DIR   = os.getcwd()  # folder where this notebook is located
m1 = pd.read_parquet(FILE_M1, columns=[
    'id', 'size_class', 'area_m2',
    'elec_m1_kwh', 'heat_total_kwh', 'pv_potential_kwh'
])
m3b = pd.read_parquet(FILE_M3B, columns=[
    'id', 'size_class', 'area_m2',
    'elec_m3b_kwh', 'heat_total_kwh', 'pv_potential_kwh'
])
print(f'M1  loaded: {len(m1):,} buildings')
print(f'M3b loaded: {len(m3b):,} buildings')

M1  loaded: 4,133,323 buildings
M3b loaded: 4,133,323 buildings


---
## Step 3 — Outlier Removal

In [4]:
def remove_outliers(df):
    before = len(df)
    keep = pd.Series(True, index=df.index)
    for sc in SIZE_CLASSES:
        mask = df['size_class'] == sc
        thr  = df.loc[mask, 'area_m2'].quantile(0.995)
        keep &= ~((df['size_class'] == sc) & (df['area_m2'] > thr))
    df = df[keep].copy()
    df = df.dropna(subset=[c for c in df.columns if 'kwh' in c.lower()])
    print(f'  Removed: {before - len(df):,}  Retained: {len(df):,}')
    return df

print('M1:')
m1 = remove_outliers(m1)
print('M3b:')
m3b = remove_outliers(m3b)

M1:
  Removed: 20,669  Retained: 4,112,654
M3b:
  Removed: 20,669  Retained: 4,112,654


---
## Step 4 — Compute Silhouette Curves for All 12 Cases

For each building type and variable, compute silhouette scores
for k=2 to k=10 using a 30,000 sample.
Also record the elbow k for each case.

This takes a few minutes — runs 12 × 9 = 108 clustering fits per method.

In [ ]:
def compute_silhouette_curve(values, label):
    """Returns k_range, silhouettes, elbow_k for one variable."""
    n   = len(values)
    cap = np.percentile(values, 99.9)
    vals = np.clip(values, 0, cap)
    X_all = StandardScaler().fit_transform(vals.reshape(-1, 1))

    np.random.seed(RANDOM_STATE)
    idx = np.random.choice(n, min(SAMPLE_N, n), replace=False)
    X_s = X_all[idx]

    k_range = list(range(K_MIN, min(K_MAX, n // 10) + 1))
    inertias, silhouettes = [], []

    for k in k_range:
        km  = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE,
                              n_init=10, batch_size=5_000)
        lbl = km.fit_predict(X_s)
        inertias.append(km.inertia_)
        silhouettes.append(silhouette_score(X_s, lbl,
                           sample_size=min(5_000, len(idx))))

    # Elbow
    x1,y1 = k_range[0], inertias[0]
    x2,y2 = k_range[-1], inertias[-1]
    dx,dy = x2-x1, y2-y1
    line_len = np.sqrt(dx**2 + dy**2)
    dists = [abs(dy*k - dx*i + x2*y1 - y2*x1)/line_len
             for k,i in zip(k_range, inertias)]
    elbow_k = k_range[np.argmax(dists)]

    return k_range, silhouettes, elbow_k

print('compute_silhouette_curve() ready')

In [ ]:
VARIABLES_M1  = [('elec_m1_kwh',  'Electricity'), ('heat_total_kwh', 'Heat'), ('pv_potential_kwh', 'PV')]
VARIABLES_M3B = [('elec_m3b_kwh', 'Electricity'), ('heat_total_kwh', 'Heat'), ('pv_potential_kwh', 'PV')]

results = {}  # {method: {sc: {var: (k_range, silhouettes, elbow_k)}}}

for method, df, variables in [('M1', m1, VARIABLES_M1), ('M3b', m3b, VARIABLES_M3B)]:
    results[method] = {}
    for sc in SIZE_CLASSES:
        results[method][sc] = {}
        subset = df[df['size_class'] == sc]
        for col, label in variables:
            print(f'  {method} {sc} {label}...')
            k_range, silhouettes, elbow_k = compute_silhouette_curve(
                subset[col].values.astype(float), label
            )
            results[method][sc][label] = (k_range, silhouettes, elbow_k)

print('\nAll curves computed')

---
## Step 5 — Plot Comparison Figure

One figure with 4 rows (building types) × 3 columns (variables).
Each subplot shows silhouette curves for both M1 (blue) and M3b (orange)
with elbow k marked by vertical dashed lines.

**What to look for:**
- M1 silhouette: peaks low around 0.55 then drops or flattens — normal
- M3b electricity silhouette: keeps rising toward 1.0 — artificial artefact
- M3b heat/PV silhouette: same as M1 — identical formula

In [ ]:
BASE_DIR   = os.getcwd()  # folder where this notebook is located
fig, axes = plt.subplots(4, 3, figsize=(16, 18))
plt.rcParams.update({'font.size': 9})

variable_labels = ['Electricity', 'Heat', 'PV']

for row, sc in enumerate(SIZE_CLASSES):
    for col, var in enumerate(variable_labels):
        ax = axes[row, col]

        k_m1,  sil_m1,  elbow_m1  = results['M1'][sc][var]
        k_m3b, sil_m3b, elbow_m3b = results['M3b'][sc][var]

        ax.plot(k_m1,  sil_m1,  'b-o', lw=1.5, ms=4, label=f'M1  (elbow k={elbow_m1})')
        ax.plot(k_m3b, sil_m3b, 'o-',  lw=1.5, ms=4, color='darkorange',
                label=f'M3b (elbow k={elbow_m3b})')

        ax.axvline(x=elbow_m1,  color='blue',       ls='--', lw=1.2, alpha=0.6)
        ax.axvline(x=elbow_m3b, color='darkorange',  ls='--', lw=1.2, alpha=0.6)

        ax.set_title(f'{sc} — {var}', fontsize=10, fontweight='bold')
        ax.set_xlabel('k', fontsize=9)
        ax.set_ylabel('Silhouette', fontsize=9)
        ax.set_ylim(0, 1.05)
        ax.set_xticks(k_m1)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=7, loc='upper left')

        # Annotate M3b electricity artificially rising
        if var == 'Electricity' and sil_m3b[-1] > 0.85:
            ax.annotate('artificial\n(discrete\nlookup)',
                        xy=(k_m3b[-1], sil_m3b[-1]),
                        xytext=(k_m3b[-2]-1.5, sil_m3b[-1]-0.15),
                        fontsize=7, color='darkorange',
                        arrowprops=dict(arrowstyle='->', color='darkorange', lw=0.8))

plt.suptitle('Silhouette Score Comparison — Method 1 vs Method 3b\n'
             'All 12 cases (4 building types × 3 variables)\n'
             'Silhouette reported for reference only — elbow used for k-selection',
             fontsize=11, fontweight='bold', y=1.01)

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'silhouette_comparison_m1_vs_m3b.pdf')
plt.savefig(save_path, bbox_inches='tight')
plt.show()
print(f'Saved: {save_path}')

---
## Step 6 — Summary Table

Shows elbow k and silhouette preferred k for all 12 cases side by side.

In [ ]:
print('SILHOUETTE vs ELBOW SUMMARY — ALL 12 CASES')
print('='*70)
print(f'{"Building":>10} {"Variable":>14}  {"M1 elbow":>10} {"M1 sil":>8}  {"M3b elbow":>10} {"M3b sil":>8}')
print('-'*70)

for sc in SIZE_CLASSES:
    for var in variable_labels:
        k_m1,  sil_m1,  elbow_m1  = results['M1'][sc][var]
        k_m3b, sil_m3b, elbow_m3b = results['M3b'][sc][var]
        sil_pref_m1  = k_m1[np.argmax(sil_m1)]
        sil_pref_m3b = k_m3b[np.argmax(sil_m3b)]
        print(f'{sc:>10} {var:>14}  {elbow_m1:>10} {sil_pref_m1:>8}  {elbow_m3b:>10} {sil_pref_m3b:>8}')

print('='*70)
print('\nNote:')
print('  M1  silhouette preferred k=2 in all 12 cases — too coarse for simulation')
print('  M3b silhouette preferred high k for electricity — artificial discrete artefact')
print('  Elbow gave physically meaningful k=4 or k=5 in all cases')
print('  → Elbow method selected for k-selection throughout')

---
## Key Findings

**Method 1 silhouette:**
Returned k=2 in all 12 cases. This is a known limitation on skewed energy data.
Two groups are always the most separated mathematically — small vs large buildings.
But k=2 loses everything in between and is too coarse for SESMG simulation.

**Method 3b silhouette — electricity:**
Kept rising toward 1.0 — never peaked, never dropped.
This is an artefact of the CO2online lookup table assigning identical kWh values
to buildings with the same number of dwellings. Clusters look perfectly separated
not because buildings are physically different but because many have exactly the
same value. This confirmed the discrete lookup table problem with Method 3b.

**Method 3b silhouette — heat and PV:**
Behaved normally — same as Method 1. Heat and PV use identical formulas in both
methods so their silhouette behaviour is identical.

**Conclusion:**
Elbow method was used for k-selection throughout. Silhouette was reported as
reference only and confirmed the elbow decision in every case.